# Downstream Predictive Evaluation on MIMIC-IV

This notebook evaluates the archived LLM-generated rankings on the MIMIC-IV cohort. Its
order follows the manuscript: run-level top-30 stability, feature-budget analysis,
procedure-specific performance at k = 30, comparison with conventional feature selection,
and the missingness sensitivity analysis.

The classifier-robustness section and duplicated exploratory tables from earlier drafts
have been removed. Patient-level data are not distributed; see the data policy in `README.md`.

In [ ]:
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    """Locate the repository root from either the root or notebooks directory."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "notebooks").is_dir() and (candidate / ".gitignore").is_file():
            return candidate
    raise RuntimeError("Could not locate the repository root.")


PROJECT_ROOT = find_project_root()

import itertools
import json
import os
import re

import numpy as np
import pandas as pd
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from scripts.figure_export import save_submission_figure

POOL_FILE = PROJECT_ROOT / "data" / "derived" / "feature_pool.txt"
AGG_DIR = PROJECT_ROOT / "results" / "aggregated"
RAW_LLM_DIR = PROJECT_ROOT / "results" / "llm_runs"
OUT = PROJECT_ROOT / "results" / "downstream"
FIGURE_DIR = PROJECT_ROOT / "figures"
PRIVATE_DATA_DIR = PROJECT_ROOT / "data" / "private"
OUT.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "delirium"
CV_SEED = 42
N_SPLITS = 5
USE_CACHE = True

DATA_OVERRIDE = os.environ.get("MIMIC_IV_DATA_PATH")
DATA_CANDIDATES = ([Path(DATA_OVERRIDE)] if DATA_OVERRIDE else []) + [
    PRIVATE_DATA_DIR / "mimic_iv_dataset_107",
    PRIVATE_DATA_DIR / "mimic_iv_dataset_107.csv",
]
DATA_FILE = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_FILE is None:
    raise FileNotFoundError(
        "Set MIMIC_IV_DATA_PATH or place the credentialed MIMIC-IV extract "
        "at the path set by MIMIC_IV_DATA_PATH. See the data policy in README.md."
    )

POOL = [line.strip() for line in POOL_FILE.read_text(encoding="utf-8-sig").splitlines()
        if line.strip()]
POOL_SET = set(POOL)
N_POOL = len(POOL)
mimic = pd.read_csv(DATA_FILE, low_memory=False)
COLS = set(mimic.columns)

print(f"Dataset: {DATA_FILE.name} ({mimic.shape[0]:,} rows x {mimic.shape[1]} columns)")
print(f"Candidate feature pool: {N_POOL}")
print(f"Incident delirium: {int(mimic[TARGET].astype(bool).sum()):,} "
      f"({100 * mimic[TARGET].astype(bool).mean():.2f}%)")


## 1. Cohort data and feature mapping

In [ ]:
# =============================================================================
# =============================================================================
META_COLS = ['subject_id', 'hadm_id', 'stay_id', 'delirium_charttime', 'delirium',
             'deathtime', 'intime', 'outtime', 'rn', 'icu_hours']

leaked = [c for c in META_COLS if c in COLS and c != TARGET]
print("Metadata columns excluded from prediction:")
for c in leaked:
    print(f'   - {c}')

In [ ]:
# =============================================================================
# =============================================================================
FEATURE_MAPPING = {
    'Age'                                                       : 'anchor_age',
    'Sex'                                                       : 'sex',
    'Ethnicity'                                                 : 'ethnicity',
    'Insurance'                                                 : 'insurance',
    'Marital Status'                                            : 'marital_status',
    'Type Of Initial ICU Admission'                             : 'type_of_initial_icu_admission',
    'Height'                                                    : 'height',
    'Body Weight'                                               : 'weight',
    'Body Mass Index'                                           : 'body_mass_index',
    'Anemia'                                                    : 'has_anemia',
    'Alcohol Abuse'                                             : 'alcohol_abuse',
    'Cancer'                                                    : 'cancer',
    'Cardiovascular Disease'                                    : 'cardiovascular_disease',
    'Cerebrovascular Disease'                                   : 'cerebrovascular_disease',
    'Charlson Comorbidity Index'                                : 'charlson_comorbidity_index',
    'Chronic Obstructive Pulmonary Disease'                     : 'copd',
    'Dementia'                                                  : 'dementia',
    'Depression'                                                : 'depression',
    'Diabetes'                                                  : 'diabetes',
    'Heart Failure'                                             : 'heart_failure',
    'Hypertension'                                              : 'hypertension',
    'Liver Disease'                                             : 'liver_disease',
    'Nicotine Dependence'                                       : 'nicotine_dependence',
    'Peripheral Vascular Disease'                               : 'peripheral_vascular_disease',
    'Renal Disease'                                             : 'renal_disease',
    'Acute Myocardial Infarction'                               : 'acute_myocardial_infarction',
    'Stroke'                                                    : 'stroke',
    'Aortic Surgery'                                            : 'aortic_surgery',
    'Coronary Artery Bypass Grafting'                           : 'cabg',
    'Heart Valve Surgery'                                       : 'heart_valve_surgery',
    'Elective Surgery'                                          : 'elective_surgery',
    'Mechanical Ventilation'                                    : 'mechanical_ventilation',
    'Renal Replacement Therapy'                                 : 'renal_replacement_therapy',
    'Acute Physiology and Chronic Health Evaluation (APACHE) Score': 'apache_score',
    'Glasgow Coma Scale'                                        : 'gcs_min',
    'Logistic Organ Dysfunction Score'                          : 'logistic_organ_dysfunction_score',
    'Oxford Acute Severity Of Illness Score'                    : 'oxford_acute_severity_of_illness_score',
    'Sequential Organ Failure Assessment Score'                 : 'sofa_score',
    'Simplified Acute Physiology Score II'                      : 'simplified_acute_physiology_score_ii',
    'Systemic Inflammatory Response Syndrome'                   : 'systemic_inflammatory_response_syndrome',
    'Sepsis'                                                    : 'sepsis',
    'Infection'                                                 : 'infection',
    'Gram Positive'                                             : 'micro_gram_positive',
    'Gram Negative'                                             : 'micro_gram_negative',
    'Type Of Microorganism'                                     : 'micro_category',
    'Antibiotic Drug Use'                                       : 'antibiotic_drug_use',
    'Diastolic Blood Pressure'                                  : ['diastolic_blood_pressure_min', 'diastolic_blood_pressure_max'],
    'Systolic Blood Pressure'                                   : ['systolic_blood_pressure_min', 'systolic_blood_pressure_max'],
    'Mean Blood Pressure'                                       : ['mean_blood_pressure_min', 'mean_blood_pressure_max'],
    'Heart Rate'                                                : ['heart_rate_min', 'heart_rate_max'],
    'Oxygen Saturation'                                         : ['oxygen_saturation_min', 'oxygen_saturation_max'],
    'Respiratory Rate'                                          : ['respiratory_rate_min', 'respiratory_rate_max'],
    'Temperature'                                               : ['temperature_min', 'temperature_max'],
    'Hematocrit'                                                : ['hematocrit_min', 'hematocrit_max'],
    'Hemoglobin'                                                : ['hemoglobin_min', 'hemoglobin_max'],
    'Platelet Count'                                            : ['platelet_count_min', 'platelet_count_max'],
    'Red Blood Cell Count'                                      : ['rbc_min', 'rbc_max'],
    'Red Cell Distribution Width'                               : ['rdw_min', 'rdw_max'],
    'Mean Corpuscular Volume'                                   : ['mcv_min', 'mcv_max'],
    'Mean Corpuscular Hemoglobin'                               : ['mch_min', 'mch_max'],
    'Mean Corpuscular Hemoglobin Concentration'                 : ['mchc_min', 'mchc_max'],
    'White Blood Cell Count'                                    : ['white_blood_cell_count_min', 'white_blood_cell_count_max'],
    'Lymphocyte Count'                                          : ['lymphocyte_count_min', 'lymphocyte_count_max'],
    'Monocyte Count'                                            : ['monocyte_count_min', 'monocyte_count_max'],
    'Neutrophil Count'                                          : ['neutrophil_count_min', 'neutrophil_count_max'],
    'Blood Sodium'                                              : ['blood_sodium_min', 'blood_sodium_max'],
    'Potassium'                                                 : ['potassium_min', 'potassium_max'],
    'Chloride'                                                  : ['chloride_min', 'chloride_max'],
    'Calcium'                                                   : ['calcium_min', 'calcium_max'],
    'Magnesium'                                                 : ['magnesium_min', 'magnesium_max'],
    'Phosphate'                                                 : ['phosphate_min', 'phosphate_max'],
    'Bicarbonate'                                               : ['bicarbonate_min', 'bicarbonate_max'],
    'Anion Gap'                                                 : ['anion_gap_min', 'anion_gap_max'],
    'Blood Urea Nitrogen'                                       : ['blood_urea_nitrogen_min', 'blood_urea_nitrogen_max'],
    'Creatinine'                                                : ['creatinine_min', 'creatinine_max'],
    'Glucose'                                                   : ['glucose_min', 'glucose_max'],
    'Albumin'                                                   : ['albumin_min', 'albumin_max'],
    'Bilirubin'                                                 : ['bilirubin_min', 'bilirubin_max'],
    'Alanine Aminotransferase'                                  : ['alanine_aminotransferase_min', 'alanine_aminotransferase_max'],
    'Aspartate Aminotransferase'                                : ['aspartate_aminotransferase_min', 'aspartate_aminotransferase_max'],
    'Alkaline Phosphatase'                                      : ['alkaline_phosphatase_min', 'alkaline_phosphatase_max'],
    'Prothrombin Time'                                          : ['prothrombin_time_min', 'prothrombin_time_max'],
    'Partial Thromboplastin Time'                               : ['partial_thromboplastin_time_min', 'partial_thromboplastin_time_max'],
    'International Normalized Ratio'                            : ['international_normalized_ratio_min', 'international_normalized_ratio_max'],
    'Blood pH'                                                  : ['blood_ph_min', 'blood_ph_max'],
    'Arterial Partial Pressure Of Oxygen'                       : ['arterial_partial_pressure_of_oxygen_min', 'arterial_partial_pressure_of_oxygen_max'],
    'Arterial Partial Pressure Of Carbon Dioxide'               : ['arterial_partial_pressure_of_carbon_dioxide_min', 'arterial_partial_pressure_of_carbon_dioxide_max'],
    'Base Excess'                                               : ['base_excess_min', 'base_excess_max'],
    'Total Carbon Dioxide'                                      : ['total_carbon_dioxide_min', 'total_carbon_dioxide_max'],
    'Lactate'                                                   : ['lactate_min', 'lactate_max'],
    'Sedation Use'                                              : 'sedation_use',
    'Use Of Midazolam'                                          : 'use_of_midazolam',
    'Vasopressor Use'                                           : 'vasopressor_use',
    'Statin Drugs'                                              : 'statin_drugs',
    'Acute Kidney Injury'                                       : 'acute_kidney_injury',
    'Urine Output'                                              : 'urine_output',
    'Glomerular Filtration Rate'                                : 'egfr',
    'Mean Blood Glucose'                                        : 'mean_blood_glucose',
    'Mean Absolute Glucose'                                     : 'mean_absolute_glucose',
    'Mean Amplitude Of Glycemic Excursions'                     : 'mage',
    'Largest Amplitude Of Glycemic Excursions'                  : 'lage',
    'Glycemic Lability Index'                                   : 'gli',
    'Glycemic Lability Index'                                   : 'gli',
    'Geriatric Nutritional Risk Index'                          : 'gnri',
    'Prognostic Nutritional Index'                              : 'prognostic_nutritional_index',
    'Lymphocyte To Monocyte Ratio'                              : 'lymphocyte_to_monocyte_ratio',
    'Neutrophil To Lymphocyte Ratio'                            : 'neutrophil_to_lymphocyte_ratio',
    'Platelet To Lymphocyte Ratio'                              : 'platelet_to_lymphocyte_ratio',
}

In [ ]:
# =============================================================================
# =============================================================================
audit = []
for f in POOL:
    v  = FEATURE_MAPPING.get(f)
    vs = [] if v is None else ([v] if isinstance(v, str) else list(v))
    ok = [c for c in vs if c in COLS]
    audit.append({'feature': f,
                  'columns': ', '.join(vs) if vs else '',
                  'n_columns': len(ok),
                  'status': 'OK' if ok and len(ok) == len(vs) else
                            ('PARTIAL' if ok else 'MISSING')})
audit_df = pd.DataFrame(audit)

missing_keys = sorted(POOL_SET - set(FEATURE_MAPPING))
extra_keys   = sorted(set(FEATURE_MAPPING) - POOL_SET)
bad          = audit_df[audit_df.status != 'OK']

print(f"Feature pool: {N_POOL}; mapping definitions: {len(FEATURE_MAPPING)}")
print('Mapping status: ' + ' / '.join(f'{k} {v}' for k, v in audit_df.status.value_counts().items()))
if missing_keys: print('\n[Missing mappings]', missing_keys)
if extra_keys: print('[Mappings outside pool]', extra_keys)
if len(bad):     display(bad)

assert not missing_keys, f'Features missing from the mapping: {missing_keys}'
assert not extra_keys,   f'Mapping keys outside the feature pool: {extra_keys}'
assert len(bad) == 0,    'At least one feature does not map to a dataset column.'

USABLE   = list(POOL)
ALL_COLS = list(dict.fromkeys(itertools.chain.from_iterable(
    ([v] if isinstance(v, str) else list(v)) for v in FEATURE_MAPPING.values())))
assert not (set(ALL_COLS) & set(META_COLS)), 'A metadata column was included among predictors.'

audit_df.to_csv(os.path.join(OUT, 'feature_mapping_audit.csv'), index=False, encoding='utf-8-sig')
print(f'\nOK - all {N_POOL} features mapped to {len(ALL_COLS)} dataset columns '
      f'(including {sum(1 for v in FEATURE_MAPPING.values() if not isinstance(v, str))} min/max pairs)')

In [ ]:
# =============================================================================
# =============================================================================
comb = pd.read_csv(os.path.join(AGG_DIR, 'combined_ranks.csv'), encoding='utf-8-sig')
comb.columns = [str(c).strip().strip('﻿') for c in comb.columns]
feat_col = comb.columns[0]

METHOD_COLS = [c for c in comb.columns
               if c != feat_col and c != 'mean_rank'
               and re.match(r'^(rank|score|seq|lmprior)_', c)]

METHOD_VECTORS = {}
for c in METHOD_COLS:
    v = comb[[feat_col, c]].rename(columns={feat_col: 'feature', c: 'v'}).dropna()
    METHOD_VECTORS[c] = v.reset_index(drop=True)

for name, v in METHOD_VECTORS.items():
    s = set(v['feature'])
    assert s == POOL_SET, f'{name}: feature-pool mismatch (missing={sorted(POOL_SET - s)}, extra={sorted(s - POOL_SET)})'

def top_k(name, k):
    """Return the top-k feature names for one procedure."""
    return METHOD_VECTORS[name].sort_values('v').head(k)['feature'].tolist()

print(f'{len(METHOD_VECTORS)} procedure vectors validated against all {N_POOL} features')
print(sorted(METHOD_VECTORS))

## 2. Common preprocessing and XGBoost evaluation

In [ ]:
# =============================================================================
# =============================================================================
NAN_NATIVE_MODELS = {'xgboost', 'lightgbm'}
ADD_MISSING_INDICATOR = True

def features_to_columns(feats):
    cols, unavailable = [], []
    for f in feats:
        v = FEATURE_MAPPING.get(f)
        if v is None:
            unavailable.append(f); continue
        vs = [v] if isinstance(v, str) else list(v)
        ok = [c for c in vs if c in COLS]
        if ok: cols.extend(ok)
        else:  unavailable.append(f)
    return list(dict.fromkeys(cols)), unavailable


def build_xy(cols, impute):
    """Build predictors; optionally impute values and add missingness indicators."""
    X = mimic[cols].copy()
    y = mimic[TARGET].astype(int)

    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    num = [c for c in X.columns if c not in cat]
    for c in num:
        X[c] = pd.to_numeric(X[c], errors='coerce')

    if impute:
        if ADD_MISSING_INDICATOR:
            ind = {f'{c}_isna': X[c].isna().astype(int) for c in num if X[c].isna().any()}
            if ind:
                X = pd.concat([X, pd.DataFrame(ind, index=X.index)], axis=1)
        for c in num:
            X[c] = X[c].fillna(X[c].median())
        for c in cat:
            mode = X[c].mode(dropna=True)
            X[c] = X[c].fillna(mode.iloc[0] if len(mode) else 'Unknown')
    else:
        for c in cat:
            X[c] = X[c].fillna('__missing__')

    if cat:
        X = pd.get_dummies(X, columns=cat, drop_first=False)
    return X, y

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

NAN_NATIVE_MODELS = {"xgboost"}
ADD_MISSING_INDICATOR = True


def make_model(name, y_train, seed=CV_SEED):
    if name != "xgboost":
        raise ValueError(f"Unsupported downstream classifier: {name}")
    n_pos = int((y_train == 1).sum())
    n_neg = int((y_train == 0).sum())
    scale_pos_weight = n_neg / n_pos if n_pos else 1.0
    return XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="auc",
        importance_type="gain",
        random_state=seed,
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        n_jobs=-1,
    )


def cv_scores(cols, model_name="xgboost", seed=CV_SEED):
    X, y = build_xy(cols, impute=False)
    splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    auroc, auprc = [], []
    for train, test in splitter.split(X, y):
        model = make_model(model_name, y.iloc[train], seed)
        model.fit(X.iloc[train], y.iloc[train])
        prediction = model.predict_proba(X.iloc[test])[:, 1]
        auroc.append(roc_auc_score(y.iloc[test], prediction))
        auprc.append(average_precision_score(y.iloc[test], prediction))
    return np.asarray(auroc), np.asarray(auprc)


print("Representative downstream classifier: XGBoost")


In [ ]:
all_columns, unavailable = features_to_columns(POOL)
if unavailable:
    raise ValueError(f"Unmapped features: {unavailable}")
baseline_file = OUT / "Downstream_baselines.csv"
if USE_CACHE and baseline_file.exists():
    baselines = pd.read_csv(baseline_file)
else:
    auroc, auprc = cv_scores(all_columns, "xgboost")
    baselines = pd.DataFrame([{
        "model": "xgboost",
        "missing_policy": "native NaN handling",
        "n_cols": len(all_columns),
        "auc": round(float(auroc.mean()), 4),
        "auc_std": round(float(auroc.std(ddof=0)), 4),
        "ap": round(float(auprc.mean()), 4),
    }])
    baselines.to_csv(baseline_file, index=False, encoding="utf-8-sig")
baselines


In [ ]:
# =============================================================================
#
# =============================================================================
N_BOOT = 2000

def cv_oof(cols, model_name, seed=CV_SEED):
    """Return observed outcomes and pooled out-of-fold probabilities."""
    impute = model_name not in NAN_NATIVE_MODELS
    X, y   = build_xy(cols, impute=impute)
    oof    = np.zeros(len(y))
    skf    = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    for tr, te in skf.split(X, y):
        m = make_model(model_name, y.iloc[tr], seed)
        m.fit(X.iloc[tr], y.iloc[tr])
        oof[te] = m.predict_proba(X.iloc[te])[:, 1]
    return y.to_numpy(), oof


def boot_ci(y, p, B=N_BOOT, seed=CV_SEED, metric=roc_auc_score):
    rng, n, out = np.random.default_rng(seed), len(y), []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        if len(np.unique(y[idx])) < 2:
            continue
        out.append(metric(y[idx], p[idx]))
    out = np.asarray(out)
    return (float(metric(y, p)),
            float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5)))


def boot_paired(y, pA, pB, B=N_BOOT, seed=CV_SEED, metric=roc_auc_score):
    """Bootstrap the paired performance difference A - B."""
    rng, n, d = np.random.default_rng(seed), len(y), []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        if len(np.unique(y[idx])) < 2:
            continue
        d.append(metric(y[idx], pA[idx]) - metric(y[idx], pB[idx]))
    d = np.asarray(d)
    p_two = 1.0 if np.allclose(d, 0.0) else 2 * min((d > 0).mean(), (d < 0).mean())
    return (float(metric(y, pA) - metric(y, pB)),
            float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)),
            float(min(p_two, 1.0)))

print(f"Patient-level bootstrap replicates: {N_BOOT}")

In [ ]:
# =============================================================================
# =============================================================================
TOP_K = 30
_f_ci = str(OUT / 'Downstream_bootstrap_ci.csv')
OOF   = {}

for name in sorted(METHOD_VECTORS):
    cols, unavail = features_to_columns(top_k(name, TOP_K))
    assert not unavail, f'{name}: unavailable mapped features {unavail}'
    OOF[name] = cv_oof(cols, 'xgboost')
    print(f'  {name:16} out-of-fold predictions complete', flush=True)

if USE_CACHE and os.path.exists(_f_ci):
    ci_df = pd.read_csv(_f_ci)
else:
    rows = []
    for name in sorted(METHOD_VECTORS):
        y, p = OOF[name]
        auc, lo, hi   = boot_ci(y, p)
        ap, alo, ahi  = boot_ci(y, p, metric=average_precision_score)
        rows.append({'method': name, 'k': TOP_K,
                     'auroc': round(auc, 4), 'auroc_ci': f'[{lo:.4f}, {hi:.4f}]',
                     'auprc': round(ap, 4),  'auprc_ci': f'[{alo:.4f}, {ahi:.4f}]'})
    ci_df = pd.DataFrame(rows).sort_values('auroc', ascending=False).reset_index(drop=True)
    ci_df.to_csv(_f_ci, index=False, encoding='utf-8-sig')
display(ci_df)

## 3. Run-level stability of top-30 feature sets

In [ ]:
# =============================================================================
# =============================================================================
RESULTS_DIR = RAW_LLM_DIR
RUN_IDS = [1, 2, 3, 4, 5]
VALUE_FIELD = {'rank': 'rank', 'seq': 'step', 'score': 'score', 'lmprior': 'score'}
HIGHER_IS_BETTER = {'score', 'lmprior'}
_CANON = {p.lower(): p for p in POOL}
_canon = lambda s: _CANON.get(str(s).strip().lower(), str(s).strip())

LBL = {'rank': 'Ranking', 'score': 'Scoring', 'seq': 'Sequencing', 'lmprior': 'Log-prior scoring'}
SHORT = {'gpt': 'GPT-4.1', 'gemini': 'Gemini 2.5 Pro', 'claude': 'Claude Sonnet 4.5'}
_split = lambda nm: (nm.split('_')[0], nm.split('_', 1)[1])
nice   = lambda nm: f'{LBL[_split(nm)[0]]} / {SHORT[_split(nm)[1]]}'

def _read_jsonl(p):
    with open(p, encoding='utf-8') as f:
        return [json.loads(l) for l in f.read().split('\n') if l.strip()]

def run_vector(name, run):
    """Return one run-specific rank vector; lower values are better."""
    me, mk = _split(name)
    folder = 'log_prior_scoring' if me == 'lmprior' else me
    d = pd.DataFrame(_read_jsonl(os.path.join(RESULTS_DIR, folder, mk, f'run_{run:02d}.jsonl')))
    d['feature'] = d['feature'].map(_canon)
    v = d[VALUE_FIELD[me]].astype(float)
    if me in HIGHER_IS_BETTER:
        v = v.round(6).rank(method='min', ascending=False)
    return d.assign(v=v)[['feature', 'v']]

_topk = lambda vec, k: vec.sort_values('v').head(k)['feature'].tolist()

_f_run = str(OUT / 'Downstream_perrun_top30.csv')
if USE_CACHE and os.path.exists(_f_run):
    perrun_summary = pd.read_csv(_f_run)
    perrun_detail  = pd.read_csv(str(OUT / 'Downstream_perrun_top30_detail.csv'))
    print(f'[cache] {os.path.basename(_f_run)}')
else:
    rows, detail = [], []
    for name in sorted(METHOD_VECTORS):
        au, ap, sets = [], [], []
        for run in RUN_IDS:
            feats = _topk(run_vector(name, run), TOP_K)
            sets.append(set(feats))
            cols, unavail = features_to_columns(feats)
            assert not unavail, f'{name} run {run}: unavailable features {unavail}'
            y, p = cv_oof(cols, 'xgboost')
            au.append(roc_auc_score(y, p)); ap.append(average_precision_score(y, p))
            detail.append({'procedure': nice(name), 'run': run,
                           'auroc': round(au[-1], 4), 'auprc': round(ap[-1], 4)})
        y, p = OOF[name]
        au, ap = np.array(au), np.array(ap)
        pair = [len(a & b) for a, b in itertools.combinations(sets, 2)]
        rows.append({'Procedure': nice(name),
            'Mean AUROC': round(au.mean(), 4), 'SD': round(au.std(ddof=1), 4),
            'Range': f'{au.min():.4f}-{au.max():.4f}',
            'Mean AUPRC': round(ap.mean(), 4), 'SD ': round(ap.std(ddof=1), 4),
            'Range ': f'{ap.min():.4f}-{ap.max():.4f}',
            'Aggregated AUROC': round(roc_auc_score(y, p), 4),
            'Aggregated AUPRC': round(average_precision_score(y, p), 4),
            'Features common to all 5 runs': len(set.intersection(*sets)),
            'Union of 5 runs': len(set.union(*sets)),
            'Mean pairwise overlap': round(float(np.mean(pair)), 1)})
        print(f'  {nice(name):32} AUROC {au.mean():.4f} +/- {au.std(ddof=1):.4f}', flush=True)
    perrun_summary = pd.DataFrame(rows).sort_values('Mean AUROC', ascending=False)
    perrun_detail  = pd.DataFrame(detail)
    perrun_summary.to_csv(_f_run, index=False, encoding='utf-8-sig')
    perrun_detail.to_csv(str(OUT / 'Downstream_perrun_top30_detail.csv'),
                         index=False, encoding='utf-8-sig')

display(perrun_summary[['Procedure', 'Mean AUROC', 'SD', 'Range',
                        'Aggregated AUROC', 'Features common to all 5 runs',
                        'Mean pairwise overlap']])
print(f"Mean within-procedure AUROC SD: {perrun_summary['SD'].mean():.4f}")
print(f"Largest range across runs: "
      f"{perrun_detail.groupby('procedure').auroc.apply(lambda x: x.max()-x.min()).max():.4f}")
print(f"Correlation between feature overlap and AUROC SD: "
      f"{perrun_summary['Mean pairwise overlap'].corr(perrun_summary['SD']):.3f}")
print(f"Aggregated ranking above the run mean: "
      f"{int((perrun_summary['Aggregated AUROC'] > perrun_summary['Mean AUROC']).sum())} / {len(perrun_summary)}")

## 4. Predictive utility across feature budgets

In [ ]:
# =============================================================================
# =============================================================================
K_SWEEP  = [5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, N_POOL]
K_SWEEP  = sorted(set(k for k in K_SWEEP if k <= N_POOL))
N_RANDOM = 5
SWEEP_MODEL = 'xgboost'
_f_sweep = str(OUT / 'Downstream_ksweep.csv')

def _one(feats):
    cols, _ = features_to_columns(feats)
    au, ap = cv_scores(cols, SWEEP_MODEL)
    return float(au.mean()), float(au.std(ddof=0)), float(ap.mean()), len(cols)

if USE_CACHE and os.path.exists(_f_sweep):
    sweep_df = pd.read_csv(_f_sweep)
    print(f'[cache] reused {os.path.basename(_f_sweep)}')
else:
    _c = None
    for _n in sorted(METHOD_VECTORS):
        _r = METHOD_VECTORS[_n].rename(columns={'v': _n})
        _c = _r if _c is None else _c.merge(_r, on='feature')
    _c['v'] = _c[sorted(METHOD_VECTORS)].mean(axis=1)
    consensus_vec = _c[['feature', 'v']].sort_values('v')
    BOTTOM_LABEL = 'BASELINE_bottom (consensus)'
    sweep = []
    for k in K_SWEEP:
        for name in sorted(METHOD_VECTORS):
            au, sd, ap, nc = _one(top_k(name, k))
            sweep.append({'selection': name, 'family': 'llm', 'k': k, 'n_cols': nc,
                          'auc': round(au, 4), 'cv_std': round(sd, 4),
                          'draw_std': np.nan, 'ap': round(ap, 4)})

        rng = np.random.default_rng(CV_SEED + k)
        r_au, r_sd, r_ap, r_nc = [], [], [], []
        for _ in range(N_RANDOM):
            pick = list(rng.choice(POOL, size=min(k, N_POOL), replace=False))
            au, sd, ap, nc = _one(pick)
            r_au.append(au); r_sd.append(sd); r_ap.append(ap); r_nc.append(nc)
        sweep.append({'selection': 'BASELINE_random', 'family': 'baseline',
                      'k': k, 'n_cols': int(np.mean(r_nc)),
                      'auc': round(float(np.mean(r_au)), 4),
                      'cv_std':   round(float(np.mean(r_sd)), 4),
                      'draw_std': round(float(np.std(r_au, ddof=0)), 4),
                      'ap':  round(float(np.mean(r_ap)), 4)})

        au, sd, ap, nc = _one(consensus_vec.tail(k)['feature'].tolist())
        sweep.append({'selection': BOTTOM_LABEL, 'family': 'baseline',
                      'k': k, 'n_cols': nc, 'auc': round(au, 4),
                      'cv_std': round(sd, 4), 'draw_std': np.nan,
                      'ap': round(ap, 4)})
        print(f'  k={k:3d} complete', flush=True)
    sweep_df = pd.DataFrame(sweep)
    sweep_df.to_csv(_f_sweep, index=False, encoding='utf-8-sig')

pivot_k = sweep_df.pivot(index='selection', columns='k', values='auc')
display(pivot_k.round(4))

## 5. Performance at the prespecified k = 30 budget

In [ ]:
paired_file = OUT / "Downstream_paired_diff.csv"
if USE_CACHE and paired_file.exists():
    paired_df = pd.read_csv(paired_file)
    print(f"[cache] reused {paired_file.name}")
else:
    best = ci_df.iloc[0]["method"]
    rows = []
    for name in sorted(OOF):
        if name == best:
            continue
        y, comparison_prediction = OOF[name]
        _, reference_prediction = OOF[best]
        difference, low, high, p_value = boot_paired(
            y, reference_prediction, comparison_prediction
        )
        rows.append({
            "reference": best,
            "method": name,
            "diff_auroc": round(difference, 4),
            "diff_ci": f"[{low:+.4f}, {high:+.4f}]",
            "p": round(p_value, 4),
            "significant": "yes" if (low > 0 or high < 0) else "no",
        })
    paired_df = pd.DataFrame(rows).sort_values("diff_auroc").reset_index(drop=True)
    paired_df.to_csv(paired_file, index=False, encoding="utf-8-sig")
display(paired_df)


## 6. Comparison with conventional feature selection

In [ ]:
# =============================================================================
#
# =============================================================================
from sklearn.linear_model import LogisticRegression as _LR
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler as _SS
import sklearn as _sk

# -----------------------------------------------------------------------------
#
# -----------------------------------------------------------------------------
_SKVER = tuple(int(p) for p in _sk.__version__.split('.')[:2])

def _l1_logreg(C=0.05, seed=CV_SEED):
    kw = dict(solver='liblinear', C=C, class_weight='balanced', random_state=seed)
    return _LR(l1_ratio=1, **kw) if _SKVER >= (1, 8) else _LR(penalty='l1', **kw)

def _assert_is_l1(fitted, tag='lasso'):
    """Confirm that L1 regularisation produced at least one zero coefficient."""
    nz = int((fitted.coef_[0] == 0).sum())
    assert nz > 0, (f'[{tag}] L1 regularisation was not applied '
                    f'({nz} zero coefficients). Check the version branch for sklearn {_sk.__version__}: '
                    f'_l1_logreg.')

def concept_to_model_columns(X):
    """Map each of the 107 clinical feature concepts to its model-input columns."""
    groups = {}
    for feature in POOL:
        raw = FEATURE_MAPPING[feature]
        raw_columns = [raw] if isinstance(raw, str) else list(raw)
        expanded = []
        for column in raw_columns:
            expanded.extend(c for c in X.columns
                            if c == column or c.startswith(f'{column}_'))
        groups[feature] = list(dict.fromkeys(expanded))

    assigned = [column for columns in groups.values() for column in columns]
    assert all(groups.values()), 'At least one clinical feature maps to no model-input column.'
    assert len(assigned) == len(set(assigned)), 'A model-input column maps to multiple concepts.'
    assert set(assigned) == set(X.columns), 'Some model-input columns are not concept-mapped.'
    return groups


def aggregate_concept_scores(column_scores, X):
    """Sum derived-column importance within each clinical feature concept."""
    score_by_column = pd.Series(np.asarray(column_scores, dtype=float), index=X.columns)
    groups = concept_to_model_columns(X)
    return pd.Series({feature: score_by_column[columns].sum()
                      for feature, columns in groups.items()})


print('Conventional selection unit: 107 clinical feature concepts')
print('Derived-column scores are summed within concept inside each training fold.')

In [ ]:
# =============================================================================
# =============================================================================
K_BUDGETS = [10, 30, 50, N_POOL]
_folds = lambda X, y: list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True,
                                           random_state=CV_SEED).split(X, y))
_FS_CACHE = {}

def fs_scores(name, fold):
    """Return feature-selection scores from one training fold."""
    key = (name, fold)
    if key in _FS_CACHE:
        return _FS_CACHE[key]
    all_cols, _ = features_to_columns(POOL)
    X, y = build_xy(all_cols, impute=False)
    tr, _ = _folds(X, y)[fold]
    Xtr, ytr = X.iloc[tr], y.iloc[tr]
    if name == 'BASELINE_lasso':
        Z = _SS().fit_transform(Xtr.fillna(Xtr.median()))
        _m = _l1_logreg(C=0.05, seed=CV_SEED).fit(Z, ytr)
        _assert_is_l1(_m, 'BASELINE_lasso')
        sc = np.abs(_m.coef_[0])
    elif name == 'BASELINE_mutual_info':
        sc = mutual_info_classif(Xtr.fillna(Xtr.median()), ytr, random_state=CV_SEED)
    elif name == 'BASELINE_xgb_gain':
        sc = make_model('xgboost', ytr, CV_SEED).fit(Xtr, ytr).feature_importances_
    else:
        raise ValueError(name)
    _FS_CACHE[key] = sc
    return sc

def cv_oof_nested_k(k, name):
    """Evaluate top-k clinical feature concepts selected within each training fold."""
    if not 1 <= k <= N_POOL:
        raise ValueError(f'k must be between 1 and {N_POOL}; received {k}.')
    all_cols, _ = features_to_columns(POOL)
    X, y = build_xy(all_cols, impute=False)
    concept_columns = concept_to_model_columns(X)
    oof, picked = np.zeros(len(y)), []
    for fi, (tr, te) in enumerate(_folds(X, y)):
        if k == N_POOL:
            selected = list(POOL)
        else:
            scores = aggregate_concept_scores(fs_scores(name, fi), X)
            selected = scores.sort_values(ascending=False, kind='stable').head(k).index.tolist()
        selected_set = set(selected)
        cols = [column for column in X.columns
                if any(column in concept_columns[feature] for feature in selected_set)]
        assert len(selected) == k and len(selected_set) == k
        if k == N_POOL:
            assert selected_set == POOL_SET and cols == X.columns.tolist()
        picked.append(selected_set)
        m = make_model('xgboost', y.iloc[tr], CV_SEED).fit(X.iloc[tr][cols], y.iloc[tr])
        oof[te] = m.predict_proba(X.iloc[te][cols])[:, 1]
    return y.to_numpy(), oof, len(set.intersection(*picked))

FS_NAMES = {'BASELINE_lasso': 'L1-regularised logistic regression',
            'BASELINE_mutual_info': 'Mutual information',
            'BASELINE_xgb_gain': 'XGBoost gain importance'}
print('Clinical-feature budgets:', K_BUDGETS, '| conventional selectors:', list(FS_NAMES))

In [ ]:
# =============================================================================
# =============================================================================
_f_ck = str(OUT / 'Downstream_conventional_ksweep.csv')
_f_cp = str(OUT / 'Downstream_conventional_paired.csv')

_cache_valid = False
if USE_CACHE and os.path.exists(_f_ck) and os.path.exists(_f_cp):
    conv_df = pd.read_csv(_f_ck); paired_k = pd.read_csv(_f_cp)
    _cache_valid = ('Selection unit' in conv_df.columns and
                    set(conv_df['Selection unit'].dropna()) == {'clinical feature concept'} and
                    set(K_BUDGETS).issubset(set(conv_df.k)))
    if _cache_valid:
        print(f'[cache] {os.path.basename(_f_ck)}')
if not _cache_valid:
    rows, pairs, OOF_K = [], [], {}
    _full_pool_oof, _full_pool_stats = None, None
    for k in K_BUDGETS:
        for name in sorted(METHOD_VECTORS):
            cols, unavail = features_to_columns(top_k(name, k))
            assert not unavail, f'{name}: unavailable mapped features {unavail}'
            if k == N_POOL:
                assert set(top_k(name, k)) == POOL_SET and set(cols) == set(all_columns)
                if _full_pool_oof is None:
                    _full_pool_oof = cv_oof(all_columns, 'xgboost')
                y, p = _full_pool_oof
            else:
                y, p = OOF[name] if k == TOP_K else cv_oof(cols, 'xgboost')
            OOF_K[(k, name)] = (y, p)
            if k == N_POOL and _full_pool_stats is not None:
                au, alo, ahi, ap, plo, phi = _full_pool_stats
            else:
                au, alo, ahi = boot_ci(y, p)
                ap, plo, phi = boot_ci(y, p, metric=average_precision_score)
                if k == N_POOL:
                    _full_pool_stats = (au, alo, ahi, ap, plo, phi)
            rows.append({'k': k, 'Selection': nice(name), 'Family': 'LLM-based',
                'AUROC': round(au, 4), 'AUROC 95% CI': f'[{alo:.4f}, {ahi:.4f}]',
                'AUPRC': round(ap, 4), 'AUPRC 95% CI': f'[{plo:.4f}, {phi:.4f}]',
                'Features common to all 5 folds': np.nan,
                'Selection unit': 'clinical feature concept'})
        for key, label in FS_NAMES.items():
            if k == N_POOL:
                y, p, stable = (*_full_pool_oof, N_POOL)
            else:
                y, p, stable = cv_oof_nested_k(k, key)
            OOF_K[(k, key)] = (y, p)
            if k == N_POOL:
                au, alo, ahi, ap, plo, phi = _full_pool_stats
            else:
                au, alo, ahi = boot_ci(y, p)
                ap, plo, phi = boot_ci(y, p, metric=average_precision_score)
            rows.append({'k': k, 'Selection': label, 'Family': 'Conventional',
                'AUROC': round(au, 4), 'AUROC 95% CI': f'[{alo:.4f}, {ahi:.4f}]',
                'AUPRC': round(ap, 4), 'AUPRC 95% CI': f'[{plo:.4f}, {phi:.4f}]',
                'Features common to all 5 folds': stable,
                'Selection unit': 'clinical feature concept'})
            print(f'  k={k:3d} {label:36} AUROC {au:.4f}  AUPRC {ap:.4f}', flush=True)

        best_llm = max((n for n in METHOD_VECTORS),
                       key=lambda n: roc_auc_score(*OOF_K[(k, n)]))
        for key, label in FS_NAMES.items():
            y, pA = OOF_K[(k, key)]; _, pB = OOF_K[(k, best_llm)]
            for mlab, mt in [('AUROC', roc_auc_score), ('AUPRC', average_precision_score)]:
                if k == N_POOL:
                    dd, lo, hi, pv = 0.0, 0.0, 0.0, 1.0
                else:
                    dd, lo, hi, pv = boot_paired(y, pA, pB, metric=mt)
                pairs.append({'k': k, 'Metric': mlab, 'Conventional': label,
                    'Best LLM-based': nice(best_llm), 'Difference': round(dd, 4),
                    '95% CI': f'[{lo:+.4f}, {hi:+.4f}]', 'p': round(pv, 4),
                    'Significant': 'Yes' if (lo > 0 or hi < 0) else 'No'})
    conv_df  = pd.DataFrame(rows).sort_values(['k', 'AUROC'], ascending=[True, False])
    paired_k = pd.DataFrame(pairs).sort_values(['k', 'Metric', 'Difference'],
                                               ascending=[True, True, False])
    full_pool_rows = conv_df[conv_df.k == N_POOL]
    assert len(full_pool_rows) == len(METHOD_VECTORS) + len(FS_NAMES)
    assert full_pool_rows.AUROC.nunique() == 1 and full_pool_rows.AUPRC.nunique() == 1, (
        'At k=107 all procedures must use the identical full feature pool.')
    assert set(full_pool_rows.loc[full_pool_rows.Family == 'Conventional',
                              'Features common to all 5 folds']) == {N_POOL}
    conv_df.to_csv(_f_ck, index=False, encoding='utf-8-sig')
    paired_k.to_csv(_f_cp, index=False, encoding='utf-8-sig')

for k in K_BUDGETS:
    print(f'\n--- k = {k} ---')
    display(conv_df[conv_df.k == k][['Selection', 'Family', 'AUROC', 'AUROC 95% CI',
                                     'AUPRC', 'AUPRC 95% CI']])
display(paired_k)

_best_conv = conv_df[conv_df.Family == 'Conventional'].groupby('k').AUROC.max()
_best_llm  = conv_df[conv_df.Family == 'LLM-based'].groupby('k').AUROC.max()
print('\nBest conventional selector versus best LLM procedure (AUROC):')
for k in K_BUDGETS:
    print(f'  k={k:3d}  {_best_conv[k]:.4f} vs {_best_llm[k]:.4f}  ->  {_best_conv[k]-_best_llm[k]:+.4f}')

## 7. Missingness sensitivity analysis

In [ ]:
# =============================================================================
# =============================================================================
y_all = mimic[TARGET].astype(int)

rows = []
for f in POOL:
    v    = FEATURE_MAPPING[f]
    col  = v if isinstance(v, str) else v[0]
    s    = mimic[col]
    miss = s.isna()
    if miss.sum() == 0 or miss.all():
        rate = 100 * miss.mean()
        rows.append({'feature': f, 'column': col, 'missing_pct': round(rate, 1),
                     'delirium_if_missing': np.nan, 'delirium_if_observed': np.nan,
                     'sofa_if_missing': np.nan, 'sofa_if_observed': np.nan})
        continue
    rows.append({
        'feature': f, 'column': col,
        'missing_pct': round(100 * miss.mean(), 1),
        'delirium_if_missing':  round(100 * y_all[miss].mean(), 1),
        'delirium_if_observed': round(100 * y_all[~miss].mean(), 1),
        'sofa_if_missing':      round(mimic.loc[miss,  'sofa_score'].mean(), 2),
        'sofa_if_observed':     round(mimic.loc[~miss, 'sofa_score'].mean(), 2),
    })

miss_df = (pd.DataFrame(rows)
             .assign(delirium_gap=lambda d: (d.delirium_if_observed - d.delirium_if_missing).round(1),
                     sofa_gap=lambda d: (d.sofa_if_observed - d.sofa_if_missing).round(2))
             .sort_values('missing_pct', ascending=False)
             .reset_index(drop=True))
miss_df.to_csv(str(OUT / 'missingness_diagnostics.csv'), index=False, encoding='utf-8-sig')

print('Top 15 features by missingness: missing versus observed groups')
display(miss_df.head(15)[['feature', 'missing_pct', 'delirium_if_missing',
                          'delirium_if_observed', 'sofa_if_missing', 'sofa_if_observed']])

hi = miss_df[miss_df.missing_pct >= 20]
print(f'\nAmong {len(hi)} features with at least 20% missingness, observed values had a higher delirium rate for '
      f'{int((hi.delirium_gap > 0).sum())} features')
print('A positive gap indicates a higher event rate among patients who were tested.')
print('Median imputation alone may remove this missingness signal.')
for th in (30, 50, 70):
    print(f'   Missingness >= {th}%: {int((miss_df.missing_pct >= th).sum())} features')

In [ ]:
# =============================================================================
# =============================================================================
_f_sens = str(OUT / 'Downstream_missing_sensitivity.csv')
if USE_CACHE and os.path.exists(_f_sens):
    sens_df = pd.read_csv(_f_sens)
    print(f'[cache] {os.path.basename(_f_sens)}')
else:
    rows = []
    for th in [100, 70, 50, 30]:
        keep = list(POOL) if th == 100 else miss_df[miss_df.missing_pct < th]['feature'].tolist()
        cols, _ = features_to_columns(keep)
        au, ap = cv_scores(cols, 'xgboost')
        rows.append({'threshold': 'Full pool (primary)' if th == 100 else f'Missingness < {th}%',
                     'n_features': len(keep), 'n_cols': len(cols),
                     'auc': round(au.mean(), 4), 'auc_std': round(au.std(ddof=0), 4),
                     'ap': round(ap.mean(), 4)})
        print(f'  Missingness < {th}%: {len(keep):3d} features, AUROC {au.mean():.4f}', flush=True)
    sens_df = pd.DataFrame(rows)
    sens_df.to_csv(_f_sens, index=False, encoding='utf-8-sig')
display(sens_df)
print('\nOutput files:')
for f in sorted(os.listdir(OUT)):
    print('  -', f)

## 8. Manuscript-ready figures

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FormatStrFormatter
import numpy as np
import pandas as pd


FIGURE_SOURCE_DIR = OUT
FINAL_FIGURE_DIR = FIGURE_DIR

COLORS = {
    "llm": "#000000",
    "conventional": "#000000",
    "random": "#4D4D4D",
    "bottom": "#8C8C8C",
    "full": "#000000",
    "grid": "#D9D9D9",
}

MANUSCRIPT_STYLE = {
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 8,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 7.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}


def parse_ci(value):
    '''Return the lower and upper limits from strings such as '[0.7876, 0.8041]'.'''
    numbers = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", str(value))
    if len(numbers) < 2:
        raise ValueError(f"Could not parse confidence interval: {value!r}")
    return float(numbers[0]), float(numbers[1])


def save_manuscript_figure(fig, stem):
    '''Save a font-embedded vector PDF and a compliant high-resolution PNG.'''
    return save_submission_figure(
        fig, FINAL_FIGURE_DIR / stem,
        raster_kind='line_drawing', column_width='full',
    )

### Figure 4. Predictive utility across feature budgets

In [ ]:
ksweep = pd.read_csv(FIGURE_SOURCE_DIR / "Downstream_ksweep.csv")

expected_k = [5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 107]
llm_raw = ksweep.loc[ksweep["family"].eq("llm")].copy()
llm_counts = llm_raw.groupby("k")["selection"].nunique()
if llm_counts.reindex(expected_k).isna().any() or not llm_counts.eq(10).all():
    raise ValueError("Figure 4 expects ten LLM procedures at every feature budget.")

llm_summary = (
    llm_raw.groupby("k", as_index=False)
    .agg(
        auc_mean=("auc", "mean"),
        auc_min=("auc", "min"),
        auc_max=("auc", "max"),
        ap_mean=("ap", "mean"),
        ap_min=("ap", "min"),
        ap_max=("ap", "max"),
    )
    .sort_values("k")
)
random_ref = (
    ksweep.loc[ksweep["selection"].eq("BASELINE_random")]
    .sort_values("k")
    .reset_index(drop=True)
)
bottom_ref = (
    ksweep.loc[ksweep["selection"].eq("BASELINE_bottom (consensus)")]
    .sort_values("k")
    .reset_index(drop=True)
)

if (
    llm_summary["k"].tolist() != expected_k
    or random_ref["k"].tolist() != expected_k
    or bottom_ref["k"].tolist() != expected_k
):
    raise ValueError("Figure 4 inputs do not contain the expected feature budgets.")

with pd.ExcelWriter(FINAL_FIGURE_DIR / 'Figure4_feature_budget.xlsx') as writer:
    llm_summary.to_excel(writer, sheet_name='LLM summary', index=False)
    random_ref.to_excel(writer, sheet_name='Random reference', index=False)
    bottom_ref.to_excel(writer, sheet_name='Bottom reference', index=False)

with plt.rc_context(MANUSCRIPT_STYLE):
    fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.35), sharex=True)
    panel_specs = [
        ("auc", "AUROC", "(a)"),
        ("ap", "AUPRC", "(b)"),
    ]

    for panel_index, (metric, ylabel, panel_label) in enumerate(panel_specs):
        ax = axes[panel_index]
        x = llm_summary["k"].to_numpy()
        mean = llm_summary[f"{metric}_mean"].to_numpy()
        lower = llm_summary[f"{metric}_min"].to_numpy()
        upper = llm_summary[f"{metric}_max"].to_numpy()

        ax.fill_between(
            x,
            lower,
            upper,
            color=COLORS["llm"],
            alpha=0.40,
            linewidth=0,
            label="Range across 10 LLM procedures" if panel_index == 0 else "_nolegend_",
            zorder=1,
        )
        ax.plot(
            x,
            mean,
            color=COLORS["llm"],
            marker="o",
            markersize=3.8,
            linewidth=1.8,
            label="LLM procedures (mean)" if panel_index == 0 else "_nolegend_",
            zorder=3,
        )
        ax.plot(
            random_ref["k"],
            random_ref[metric],
            color=COLORS["random"],
            marker="s",
            markerfacecolor="white",
            markeredgewidth=0.9,
            markersize=3.8,
            linewidth=1.25,
            linestyle="--",
            label="Random selection" if panel_index == 0 else "_nolegend_",
            zorder=2,
        )
        ax.plot(
            bottom_ref["k"],
            bottom_ref[metric],
            color=COLORS["bottom"],
            marker="^",
            markerfacecolor="white",
            markeredgewidth=0.9,
            markersize=4.0,
            linewidth=1.25,
            linestyle="-.",
            label="Bottom-k consensus" if panel_index == 0 else "_nolegend_",
            zorder=2,
        )

        ax.axvline(30, color="#666666", linewidth=0.9, linestyle=":", zorder=0)
        ax.text(
            30,
            0.985,
            "k = 30",
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="top",
            color="#666666",
            fontsize=7.2,
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.78, "pad": 1.0},
        )
        ax.text(
            0.01,
            0.98,
            panel_label,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontweight="bold",
        )
        ax.set_xlabel("Number of selected features (k)")
        ax.set_ylabel(ylabel)
        ax.set_xticks(expected_k)
        ax.grid(axis="y", color=COLORS["grid"], linewidth=0.6, alpha=0.65)
        ax.spines[["top", "right"]].set_visible(False)
        ax.margins(x=0.012)

    handles, labels = axes[0].get_legend_handles_labels()
    order = [1, 0, 2, 3]
    fig.legend(
        [handles[i] for i in order],
        [labels[i] for i in order],
        loc="upper center",
        bbox_to_anchor=(0.5, 1.04),
        ncol=4,
        frameon=False,
        handlelength=2.4,
        columnspacing=1.3,
    )
    fig.subplots_adjust(top=0.84, bottom=0.17, left=0.08, right=0.99, wspace=0.25)

    save_manuscript_figure(fig, "Figure4_feature_budget")
    plt.show()
    plt.close(fig)

### Figure 5. Performance at k = 30

In [ ]:
forest = pd.read_csv(FIGURE_SOURCE_DIR / "Downstream_conventional_ksweep.csv")
forest = forest.loc[forest["k"].eq(30)].copy()
forest["Selection"] = forest["Selection"].replace(
    {"Log-prior / GPT-4.1": "Log-prior scoring / GPT-4.1"}
)
forest = forest.sort_values("AUROC", ascending=False).reset_index(drop=True)

if len(forest) != 13:
    raise ValueError(f"Figure 5 expects 13 procedures at k = 30; found {len(forest)}.")
if forest["Selection"].nunique() != 13:
    raise ValueError("Figure 5 procedure labels are not unique.")
if forest["Family"].value_counts().to_dict() != {"LLM-based": 10, "Conventional": 3}:
    raise ValueError("Figure 5 expects ten LLM-based and three conventional procedures.")

for metric in ("AUROC", "AUPRC"):
    parsed = forest[f"{metric} 95% CI"].map(parse_ci)
    forest[f"{metric}_low"] = parsed.map(lambda value: value[0])
    forest[f"{metric}_high"] = parsed.map(lambda value: value[1])

reference_source = pd.read_csv(FIGURE_SOURCE_DIR / "Downstream_ksweep.csv")
reference_k30 = reference_source.loc[reference_source["k"].eq(30)].set_index("selection")
full_pool_source = pd.read_csv(FIGURE_SOURCE_DIR / "Downstream_baselines.csv")
full_pool = full_pool_source.loc[full_pool_source["model"].eq("xgboost")]
if len(full_pool) != 1:
    raise ValueError("Expected one full-pool XGBoost reference row.")

references = {
    "AUROC": {
        "Full pool": float(full_pool.iloc[0]["auc"]),
        "Random-30": float(reference_k30.loc["BASELINE_random", "auc"]),
        "Bottom-30 consensus": float(
            reference_k30.loc["BASELINE_bottom (consensus)", "auc"]
        ),
    },
    "AUPRC": {
        "Full pool": float(full_pool.iloc[0]["ap"]),
        "Random-30": float(reference_k30.loc["BASELINE_random", "ap"]),
        "Bottom-30 consensus": float(
            reference_k30.loc["BASELINE_bottom (consensus)", "ap"]
        ),
    },
}

family_styles = {
    "LLM-based": {
        "color": COLORS["llm"],
        "marker": "o",
        "markerfacecolor": "white",
    },
    "Conventional": {
        "color": COLORS["conventional"],
        "marker": "D",
        "markerfacecolor": "black",
    },
}
reference_rows = [
    {'Metric': metric, 'Reference': name, 'Value': value}
    for metric, values in references.items()
    for name, value in values.items()
]
with pd.ExcelWriter(FINAL_FIGURE_DIR / 'Figure5_k30_forest.xlsx') as writer:
    forest.to_excel(writer, sheet_name='Procedures at k=30', index=False)
    pd.DataFrame(reference_rows).to_excel(writer, sheet_name='References', index=False)

reference_styles = {
    "Full pool": {"color": COLORS["full"], "linestyle": "-", "linewidth": 1.15},
    "Random-30": {"color": COLORS["random"], "linestyle": "--", "linewidth": 1.05},
    "Bottom-30 consensus": {
        "color": COLORS["bottom"],
        "linestyle": ":",
        "linewidth": 1.15,
    },
}

with plt.rc_context(MANUSCRIPT_STYLE):
    fig, axes = plt.subplots(1, 2, figsize=(9.4, 5.55), sharey=True)
    y = np.arange(len(forest))

    for panel_index, metric in enumerate(("AUROC", "AUPRC")):
        ax = axes[panel_index]
        for family, style in family_styles.items():
            mask = forest["Family"].eq(family)
            rows = forest.loc[mask]
            y_family = y[mask.to_numpy()]
            point = rows[metric].to_numpy()
            low = rows[f"{metric}_low"].to_numpy()
            high = rows[f"{metric}_high"].to_numpy()
            ax.errorbar(
                point,
                y_family,
                xerr=np.vstack((point - low, high - point)),
                fmt=style["marker"],
                color=style["color"],
                ecolor=style["color"],
                markersize=5.0,
                markerfacecolor=style["markerfacecolor"],
                markeredgecolor="black",
                markeredgewidth=0.9,
                elinewidth=1.1,
                capsize=2.1,
                linestyle="none",
                zorder=4,
            )

        for reference_name, value in references[metric].items():
            ax.axvline(value, zorder=1, **reference_styles[reference_name])

        ax.set_title(f"({'ab'[panel_index]}) {metric}", loc="left", fontweight="bold")
        ax.set_xlabel(metric)
        ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        ax.grid(axis="x", color=COLORS["grid"], linewidth=0.6, alpha=0.65, zorder=0)
        ax.spines[["top", "right", "left"]].set_visible(False)
        ax.tick_params(axis="y", length=0)

    axes[0].set_xlim(0.70, 0.85)
    axes[1].set_xlim(0.18, 0.36)
    axes[0].set_yticks(y)
    axes[0].set_yticklabels(forest["Selection"])
    axes[0].invert_yaxis()

    for tick_label, family in zip(axes[0].get_yticklabels(), forest["Family"]):
        tick_label.set_color(family_styles[family]["color"])
        if family == "Conventional":
            tick_label.set_fontweight("bold")

    legend_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor="white",
            markeredgecolor="black",
            markersize=6,
            label="LLM-based",
        ),
        Line2D(
            [0],
            [0],
            marker="D",
            color="none",
            markerfacecolor="black",
            markeredgecolor="black",
            markersize=6,
            label="Conventional",
        ),
        Line2D([0], [0], label="Full pool", **reference_styles["Full pool"]),
        Line2D([0], [0], label="Random selection (k = 30)", **reference_styles["Random-30"]),
        Line2D(
            [0],
            [0],
            label="Bottom-k consensus (k = 30)",
            **reference_styles["Bottom-30 consensus"],
        ),
    ]
    fig.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.61, 1.01),
        ncol=5,
        frameon=False,
        handlelength=2.4,
        columnspacing=1.25,
    )
    fig.subplots_adjust(top=0.88, bottom=0.10, left=0.37, right=0.99, wspace=0.18)

    save_manuscript_figure(fig, "Figure5_k30_forest")
    plt.show()
    plt.close(fig)